# SER - Experiment 08: early-stopping criterion (val_loss vs val_accuracy)

**Optional but high-value.** Roughly 20 minutes, and worth more than the whole
regularisation sweep.

The base run's per-epoch history shows validation loss and validation accuracy
disagreeing about which epoch is best:

| Epoch | 4 | 9 | 11 | **13** | 14 |
|---|---|---|---|---|---|
| val accuracy | 58.58% | 59.71% | 61.05% | **61.87%** | 61.77% |
| val loss | **1.1136** | 1.3231 | 1.4858 | 1.5056 | 1.6032 |

Validation loss is best at epoch 4; validation accuracy peaks at epoch 13,
**3.29 points higher**. Cross-entropy penalises growing over-confidence even
while the argmax keeps improving, so monitoring `val_loss` throws away the
more accurate model.

This notebook re-runs `base` and the sweep winner `gap_reg_aug3` with
`EarlyStopping` monitoring **`val_accuracy`** instead, and appends the results
to `sweep_results.json` so notebook 07 can pick the overall winner.

Still **validation only** - the test set is not loaded.

**Attach:** the four corpora, `ser-feature-cache`, **and notebook 06's
output**.
**Accelerator: GPU.**

In [ ]:
import glob
import json
import os
import shutil
import sys
import time

import numpy as np
import tensorflow as tf
import keras

print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
assert gpus, "Set Settings -> Accelerator -> GPU before running."
print("GPUs:", gpus)

In [ ]:
REPO = "https://github.com/Eldorado5002/ser.git"

if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser

sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
DATA_ROOT = "/kaggle/working/ser/data"

CANONICAL = {
    "RAVDESS": "audio_speech_actors_01-24",
    "TESS":    "TESS Toronto emotional speech set data",
    "SAVEE":   "ALL",
    "CREMA-D": "AudioWAV",
}


def find_canonical(target):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        for d in dirs:
            if d.lower() == target.lower():
                hits.append(os.path.join(root, d))
    return sorted(hits)[0] if hits else None


os.makedirs(DATA_ROOT, exist_ok=True)
for name, target in CANONICAL.items():
    src = find_canonical(target)
    assert src is not None, f"MISSING INPUT for {name}: no '{target}' found"
    dst = os.path.join(DATA_ROOT, name)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f"{name:9s} -> {src}")

In [ ]:
import config
from data_loader import build_metadata, split_metadata
from augmentation import plan_augmentation
from features import build_feature_matrix, df_to_items
from utils import StreamScalers, set_seed

config.CACHE_DIR = "/kaggle/working/features_cache"
config.RUNS_DIR = "/kaggle/working/runs"
os.makedirs(config.CACHE_DIR, exist_ok=True)
os.makedirs(config.RUNS_DIR, exist_ok=True)

staged = 0
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".npz"):
            shutil.copy(os.path.join(root, f), config.CACHE_DIR)
            staged += 1
print(f"staged {staged} cache files")

# Carry forward notebook 06's results so 07 sees one merged file.
prior = []
for root, dirs, files in os.walk("/kaggle/input"):
    if "sweep_results.json" in files:
        prior = json.load(open(os.path.join(root, "sweep_results.json")))
        print(f"loaded {len(prior)} prior candidates")
        break

meta = build_metadata(strict=True)
assert len(meta) == 12162
train_df, val_df, test_df = split_metadata(meta)
set_seed(config.RANDOM_SEED)


def train_features(kind):
    if kind == "3x":
        orig = config.TARGET_TRAIN_SIZE
        config.TARGET_TRAIN_SIZE = 3 * len(train_df)
        items = plan_augmentation(train_df, emotion_aware=False)
        config.TARGET_TRAIN_SIZE = orig
        return build_feature_matrix(items, desc="train_uniform3x")
    items = plan_augmentation(train_df, emotion_aware=False)
    return build_feature_matrix(items, desc="train_uniform")


val_feats = build_feature_matrix(df_to_items(val_df), desc="val")
PREPARED = {}
for kind in ("1x", "3x"):
    tf_ = train_features(kind)
    sc = StreamScalers().fit(tf_)
    PREPARED[kind] = (sc.transform(tf_),
                      tf.keras.utils.to_categorical(tf_["y"],
                                                    config.NUM_CLASSES),
                      sc.transform(val_feats),
                      tf.keras.utils.to_categorical(val_feats["y"],
                                                    config.NUM_CLASSES))
print("\nTEST SET NOT LOADED - still sealed.")

In [ ]:
from model import build_model

# Same two architectures as before; the ONLY change is the early-stopping
# and checkpoint criterion: val_accuracy (max) instead of val_loss (min).
CANDIDATES = [
    dict(tag="base_va", data="1x", lr=1e-3, patience=12, kw={}),
    dict(tag="gap_reg_aug3_va", data="3x", lr=1e-3, patience=12,
         kw=dict(head="gap", dropout_conv=0.35, dropout_dense=0.55,
                 l2=1e-4)),
]

results = list(prior)
done = {r["tag"] for r in results}
started = time.time()

for c in CANDIDATES:
    if c["tag"] in done:
        print(f"[skip] {c['tag']}")
        continue

    x_tr, y_tr, x_va, y_va = PREPARED[c["data"]]
    tf.keras.backend.clear_session()
    set_seed(config.RANDOM_SEED)

    model, _ = build_model(use_afw=False, use_mstc=False, **c["kw"])
    model.compile(optimizer=tf.keras.optimizers.Adam(c["lr"]),
                  loss="categorical_crossentropy", metrics=["accuracy"])

    print(f"\n{'=' * 72}")
    print(f"[run] {c['tag']}   {model.count_params():,} params   "
          f"monitor=val_accuracy   ({(time.time()-started)/60:.0f} min)")
    print(f"{'=' * 72}")

    hist = model.fit(
        x_tr, y_tr, validation_data=(x_va, y_va),
        epochs=config.EPOCHS, batch_size=config.BATCH_SIZE, verbose=2,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", mode="max", patience=c["patience"],
                restore_best_weights=True, verbose=1),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=config.REDUCE_LR_FACTOR,
                patience=config.REDUCE_LR_PATIENCE, min_lr=config.MIN_LR,
                verbose=0),
        ])

    h = hist.history
    best_i = int(np.argmax(h["val_accuracy"]))   # selection criterion
    rec = {
        "tag": c["tag"], "params": int(model.count_params()),
        "val_accuracy": float(h["val_accuracy"][best_i]),
        "val_loss": float(h["val_loss"][best_i]),
        "train_accuracy": float(h["accuracy"][best_i]),
        "gap": float(h["accuracy"][best_i] - h["val_accuracy"][best_i]),
        "best_epoch": best_i + 1, "epochs_run": len(h["val_accuracy"]),
        "data": c["data"], "lr": c["lr"], "monitor": "val_accuracy",
        "kw": {k: str(v) for k, v in c["kw"].items()},
    }
    results.append(rec)
    json.dump(results, open("/kaggle/working/sweep_results.json", "w"),
              indent=2)
    model.save(os.path.join(config.RUNS_DIR, f"sweep_{c['tag']}.keras"))
    print(f"\n  -> val_acc {rec['val_accuracy']*100:.2f}%   "
          f"best epoch {rec['best_epoch']}   ran {rec['epochs_run']}")

In [ ]:
rows = sorted(results, key=lambda r: -r["val_accuracy"])
print("=" * 86)
print("  ALL CANDIDATES - VALIDATION ONLY (test set untouched)")
print("=" * 86)
print(f"  {'candidate':24s} {'monitor':13s} {'val acc':>8s} {'gap':>6s} "
      f"{'params':>11s} {'epoch':>6s}")
print("  " + "-" * 82)
for r in rows:
    print(f"  {r['tag']:24s} {r.get('monitor','val_loss'):13s} "
          f"{r['val_accuracy']*100:7.2f}% {r['gap']*100:5.1f} "
          f"{r['params']:11,} {r['best_epoch']:6d}")
print("=" * 86)

base_loss = next((r for r in results if r["tag"] == "base"), None)
base_acc_m = next((r for r in results if r["tag"] == "base_va"), None)
if base_loss and base_acc_m:
    d = (base_acc_m["val_accuracy"] - base_loss["val_accuracy"]) * 100
    print(f"\n  criterion effect on base: {d:+.2f} points "
          f"(val_loss {base_loss['val_accuracy']*100:.2f}% -> "
          f"val_accuracy {base_acc_m['val_accuracy']*100:.2f}%)")

print(f"\n  overall winner: {rows[0]['tag']} at "
      f"{rows[0]['val_accuracy']*100:.2f}% validation")
print("  Run notebook 07 next; it reads this merged sweep_results.json.")

for name in CANONICAL:
    link = os.path.join(DATA_ROOT, name)
    if os.path.islink(link):
        os.unlink(link)
print("\noutput:", sorted(os.listdir("/kaggle/working")))